# Mamba vs SimpleStories Transformer — Comparative Evaluation

This notebook loads:
* A trained Mamba SSM checkpoint (from `tpu_train.py`)
* The matched SimpleStories pretrained transformer (e.g. `SimpleStories/SimpleStories-5M`)

…and runs six experiments to compare them on the SimpleStories test split:

| # | Experiment | What it measures |
|---|---|---|
| 1 | Test BPB / perplexity / top-k accuracy | Held-out language-model quality |
| 2 | Inference benchmark | Decode tok/s, prefill tok/s, state memory |
| 3 | Loss vs sequence length | Long-context extrapolation (Mamba's design pitch) |
| 4 | Per-position loss | Does the model use its context? |
| 5 | Selective copying / induction | In-context retrieval (Mamba's selectivity claim) |
| 6 | Diversity (self-BLEU / distinct-n / repetition) | Mode collapse / sample variety |

Designed to run on Google Colab (T4 / A100 GPU or CPU). All models load in fp32 by default for fair comparison.


## 1. Setup


In [ ]:
# Install dependencies. Skip if already on the host.
# - triton: required for the CUDA fused-scan kernel
# - mauve-text: distributional quality metric for Experiment 7
# - sentence-transformers: sentence-level embedding drift metric
# - accelerate + bitsandbytes: 4-bit quantization for the open-source
#   judge model in Experiment 7 (graceful fallback if unavailable)
%pip install -q torch triton transformers datasets einops tqdm matplotlib pandas nltk wandb
%pip install -q mauve-text sentence-transformers
%pip install -q accelerate bitsandbytes
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


### Get the project source

The notebook needs `mamba/mamba_llm_tpu.py` and `mamba/xla_fused_scan.py` from the
project repo to reconstruct the Mamba model. Adjust `REPO_URL` to your fork.

If you've already uploaded the project files to Colab (or are running locally), set
`REPO_DIR` to the existing path and skip the clone.


In [ ]:
import os, sys

REPO_URL = "https://github.com/dmv5383/csci739-project-mamba.git"
REPO_DIR = "csci739-project-mamba"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# If running locally from the project root, the repo is just the cwd:
if not os.path.exists(os.path.join(REPO_DIR, "mamba")):
    if os.path.exists("mamba"):
        REPO_DIR = "."
    else:
        raise FileNotFoundError("Couldn't find the mamba/ directory")

print(f"REPO_DIR = {REPO_DIR}")


In [ ]:
import math, time, json, statistics
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk import ngrams
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# CUDA setup flags (no-ops on CPU). TF32 doubles fp32 matmul throughput on
# Ampere+ with imperceptible accuracy impact; cudnn.benchmark lets cuDNN
# pick the fastest depthwise-conv1d algorithm on first call.
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

# Pick the Triton-backed CUDA path when CUDA is available, else fall back
# to the TPU/pure-PyTorch path. Both modules expose identical state_dict
# keys so a TPU-trained checkpoint loads `strict=True` either way.
if torch.cuda.is_available():
    from mamba.mamba_llm_cuda import MambaLMHeadModel, MambaLMConfig, HAS_TRITON
    print(f"Using mamba_llm_cuda (Triton-backed). Triton available: {HAS_TRITON}")
    if not HAS_TRITON:
        print("  WARNING: Triton kernel will fall back to a pure-PyTorch sequential")
        print("  scan that is unusably slow at L >= 256. Install triton, or fall back")
        print("  to mamba.mamba_llm_tpu by editing this cell.")
else:
    from mamba.mamba_llm_tpu import MambaLMHeadModel, MambaLMConfig
    HAS_TRITON = False
    print("CUDA unavailable; using mamba_llm_tpu (pure-PyTorch parallel scan).")

print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")


## 2. Configuration

Edit these to point at your specific checkpoint and matched baseline.


In [ ]:
# ─── Mamba checkpoint source ────────────────────────────────────────────
# Pick ONE: a local path, a wandb artifact, or upload to Colab and set the path.
MAMBA_CHECKPOINT_PATH = "mamba_simplestories_5m_final.pt"  # local file
MAMBA_WANDB_ARTIFACT  = None    # e.g. "entity/mamba-simplestories/mamba-5m-{run_id}:best"

# ─── SimpleStories baseline (size should match the Mamba) ───────────────
HF_TRANSFORMER_MODEL = "SimpleStories/SimpleStories-5M"
HF_TOKENIZER         = "SimpleStories/SimpleStories-5M"

# ─── Test data ──────────────────────────────────────────────────────────
DATASET_NAME    = "lennart-finke/SimpleStories"
DATASET_SPLIT   = "test"
TEXT_COLUMN     = "story"
LOWERCASE       = True       # match training preprocessing
SEQ_LEN         = 512        # trained context length
MAX_TEST_STORIES = 2000      # cap for Colab speed; set 0 for full split

# ─── Inference benchmark ────────────────────────────────────────────────
INFERENCE_DECODE_TOKENS  = 100
INFERENCE_PREFILL_LENS   = [128, 512, 1024, 2048]

# ─── Loss-vs-length test ────────────────────────────────────────────────
LONG_CONTEXT_LENS = [128, 256, 512, 1024, 2048]

# ─── Selective-copy / induction test ────────────────────────────────────
INDUCTION_DISTANCES = [4, 8, 16, 32, 64, 128, 256]
INDUCTION_TRIALS    = 500     # per distance per model
                              # 50 was too few — at vocab=4096 the random
                              # baseline gives 0.012 expected hits per 50,
                              # making "0/50" indistinguishable from a weak
                              # but real induction signal. The headline
                              # metric is 2AFC top-1 (50% baseline), which
                              # 500 trials makes statistically meaningful
                              # (95% CI half-width ≈ 4.4pp).

# ─── Diversity test ─────────────────────────────────────────────────────
DIVERSITY_PROMPTS = [
    "once upon a time,",
    "there was a little girl named",
    "the dog ran",
    "in a small village,",
]
DIVERSITY_GENS_PER_PROMPT = 20
DIVERSITY_MAX_NEW_TOKENS  = 80
DIVERSITY_TEMPERATURE     = 0.8
DIVERSITY_TOP_K           = 40

# ─── Hardware / precision ───────────────────────────────────────────────
USE_GPU = torch.cuda.is_available()
EVAL_BATCH_SIZE = 8 if USE_GPU else 2

# Quality metrics (Experiment 1 test BPB, Experiment 3 long-context loss)
# stay in fp32 to keep the cross-architecture comparison apples-to-apples.
DTYPE = torch.float32

# Inference benchmark (Experiment 2: prefill TPS / decode TPS) is reported
# at BOTH fp32 (matches the quality-metric configuration) AND the
# realistic deployment dtype — fp16 on T4/V100 (no bf16 hardware), bf16
# on Ampere+. The deployment number is what anyone serving these models
# would actually see; reporting only fp32 understates Mamba's inference
# advantage. SSM state stays fp32 either way (numerical stability — see
# `MambaBlock.allocate_inference_cache`).
if USE_GPU:
    _major = torch.cuda.get_device_capability()[0]
    INFERENCE_DTYPE_DEPLOY = torch.bfloat16 if _major >= 8 else torch.float16
else:
    INFERENCE_DTYPE_DEPLOY = torch.float32        # CPU has no half-precision matmul speedup

device = torch.device("cuda" if USE_GPU else "cpu")
print(f"Using device: {device}")
print(f"Quality metrics dtype:    {DTYPE}")
print(f"Inference benchmark also runs at: {INFERENCE_DTYPE_DEPLOY}")
torch.manual_seed(42)


## 3. Load both models


In [ ]:
def load_mamba(path=None, wandb_artifact=None):
    """Load a Mamba checkpoint produced by tpu_train.py."""
    if wandb_artifact:
        import wandb
        if wandb.run is None:
            wandb.login()
        api = wandb.Api()
        artifact = api.artifact(wandb_artifact, type="model")
        d = artifact.download()
        files = [os.path.join(d, f) for f in os.listdir(d) if f.endswith(".pt")]
        if not files:
            raise RuntimeError(f"No .pt in artifact {wandb_artifact}")
        path = files[0]
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"Mamba checkpoint not found: {path}")
    payload = torch.load(path, map_location='cpu', weights_only=False)
    cfg = MambaLMConfig(**payload['config'])
    model = MambaLMHeadModel(cfg)
    model.load_state_dict(payload['state_dict'], strict=True)
    model.to(device).to(DTYPE).eval()
    return model, cfg, payload

mamba, mamba_cfg, mamba_payload = load_mamba(MAMBA_CHECKPOINT_PATH, MAMBA_WANDB_ARTIFACT)

n_params_mamba = mamba.num_parameters(unique=True)
n_params_mamba_non_embed = n_params_mamba - mamba.embedding.weight.numel()
print(f"Mamba loaded:")
print(f"  n_layer={mamba_cfg.n_layer}  d_input={mamba_cfg.d_input}  d_model={mamba_cfg.d_model}")
print(f"  d_state={mamba_cfg.d_state}  dt_rank={mamba_cfg.dt_rank}  seq_len={mamba_cfg.seq_len}")
print(f"  n_params unique: {n_params_mamba:,}  (non-embedding: {n_params_mamba_non_embed:,})")
print(f"  checkpoint metadata: step={mamba_payload.get('global_step')}, "
      f"epoch={mamba_payload.get('epoch')}, val_loss={mamba_payload.get('val_loss')}")


In [ ]:
# Load the matched SimpleStories transformer + tokenizer.
print(f"Loading {HF_TRANSFORMER_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER)
try:
    transformer = AutoModelForCausalLM.from_pretrained(
        HF_TRANSFORMER_MODEL, torch_dtype=DTYPE,
    ).to(device).eval()
except Exception as e:
    # Some SimpleStories uploads may need trust_remote_code=True.
    print(f"Standard loader failed ({e!r}); retrying with trust_remote_code=True")
    transformer = AutoModelForCausalLM.from_pretrained(
        HF_TRANSFORMER_MODEL, torch_dtype=DTYPE, trust_remote_code=True,
    ).to(device).eval()

n_params_xfmr = sum(p.numel() for p in transformer.parameters())
print(f"Transformer loaded: {n_params_xfmr:,} params")
print(f"  config: {transformer.config}")


## 4. Unified interface

Both models need a `forward(input_ids) → logits` and a `step(input_ids, cache) → (logits, new_cache)` so the same evaluation/generation code works for either.


In [ ]:
class MambaInterface:
    name = "Mamba"
    def __init__(self, model, cfg):
        self.model, self.cfg = model, cfg

    @torch.inference_mode()
    def forward(self, input_ids):                  # (B, L) int → (B, L, V)
        return self.model(input_ids)

    def allocate_cache(self, batch_size):
        return self.model.allocate_inference_cache(
            batch_size, dtype=torch.float32, device=device,
        )

    @torch.inference_mode()
    def step(self, input_ids, cache):              # (B,) int → (B, V), cache
        return self.model.step(input_ids, cache)

    @property
    def n_params(self):
        return self.model.num_parameters(unique=True)

    def state_size_bytes(self, batch_size, seq_len):
        # Mamba state = 2 × n_layer × B × d_model × (kernel + d_state) × 4 bytes (fp32).
        # CONSTANT in seq_len — that's the whole story.
        c = self.cfg
        return 2 * c.n_layer * batch_size * c.d_model * (c.kernel_size + c.d_state) * 4


class TransformerInterface:
    name = "Transformer"
    def __init__(self, model, tokenizer):
        self.model, self.tokenizer = model, tokenizer
        self.cfg = model.config

    @torch.inference_mode()
    def forward(self, input_ids):
        return self.model(input_ids=input_ids).logits

    def allocate_cache(self, batch_size):
        return None        # HF starts decoding with past_key_values=None

    @torch.inference_mode()
    def step(self, input_ids, cache):
        outputs = self.model(
            input_ids=input_ids.unsqueeze(-1),     # (B, 1)
            past_key_values=cache,
            use_cache=True,
        )
        return outputs.logits[:, -1, :], outputs.past_key_values

    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters())

    def state_size_bytes(self, batch_size, seq_len):
        # KV cache after `seq_len` tokens:
        #   2 (k,v) × n_layer × B × n_kv × d_head × seq_len × 4 bytes (fp32)
        c = self.cfg
        n_heads = getattr(c, "num_attention_heads", None) or getattr(c, "n_head", 1)
        n_kv    = getattr(c, "num_key_value_heads", n_heads)
        d_model = getattr(c, "hidden_size", None) or getattr(c, "n_embd", n_heads * 64)
        d_head  = d_model // n_heads
        n_layer = (getattr(c, "num_hidden_layers", None) or
                   getattr(c, "n_layer", 1))
        return 2 * n_layer * batch_size * n_kv * d_head * seq_len * 4


mamba_iface = MambaInterface(mamba, mamba_cfg)
xfmr_iface  = TransformerInterface(transformer, tokenizer)
print(f"Mamba: {mamba_iface.n_params:,} params")
print(f"Transformer: {xfmr_iface.n_params:,} params")


### Sanity check — both forward + step on a tiny input


In [ ]:
test_ids = torch.tensor([[1, 2, 3, 4, 5]], device=device)
with torch.inference_mode():
    print(f"Mamba forward shape:       {mamba_iface.forward(test_ids).shape}")
    print(f"Transformer forward shape: {xfmr_iface.forward(test_ids).shape}")

# Step path
for iface in (mamba_iface, xfmr_iface):
    cache = iface.allocate_cache(batch_size=1)
    next_tok = torch.tensor([1], device=device)
    logits, cache = iface.step(next_tok, cache)
    print(f"{iface.name} step output shape: {logits.shape}")
print("Sanity OK")


## 5. Tokenize the SimpleStories test split

Same recipe as `tpu_train.py`: lowercase → fast BPE → EOS-separated stream → packed `(N, seq_len+1)` chunks. This makes test-loss numbers directly comparable to training-time val-loss numbers.


In [ ]:
print(f"Loading {DATASET_NAME} (split={DATASET_SPLIT})...")
ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
ds = ds.select_columns([TEXT_COLUMN])
if MAX_TEST_STORIES:
    ds = ds.select(range(min(MAX_TEST_STORIES, len(ds))))
print(f"  {len(ds):,} stories")

eos_id = tokenizer.eos_token_id
if eos_id is None:
    eos_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    print(f"  no EOS → using id={eos_id} as separator")

def tokenize_fn(examples):
    texts = examples[TEXT_COLUMN]
    if LOWERCASE:
        texts = [t.lower() for t in texts]
    nbytes = [len(t.encode('utf-8')) for t in texts]
    encoded = tokenizer(texts, add_special_tokens=False)['input_ids']
    return {'ids': [seq + [eos_id] for seq in encoded], 'nbytes': nbytes}

ds_tok = ds.map(tokenize_fn, batched=True, batch_size=500,
                remove_columns=ds.column_names, desc='tokenize')

stream, total_bytes = [], 0
for ex in tqdm(ds_tok, desc='concat'):
    stream.extend(ex['ids']); total_bytes += ex['nbytes']

bytes_per_token = total_bytes / max(len(stream), 1)
print(f"\nTotal tokens: {len(stream):,}  bytes/token: {bytes_per_token:.3f}")

chunk_size = SEQ_LEN + 1
n_chunks = len(stream) // chunk_size
test_data = torch.tensor(stream[:n_chunks * chunk_size], dtype=torch.long).view(n_chunks, chunk_size)
print(f"Packed test data: {test_data.shape}")


## Experiment 1 — Held-out test loss / perplexity / accuracy

Standard LM evaluation. Both models see the same packed `(x, y)` pairs and the loss is computed via mean cross-entropy. **BPB is the apples-to-apples cross-architecture metric** since both use the same tokenizer.


In [ ]:
LOG2_E = 1.0 / math.log(2.0)
def bits_per_byte(loss_nats, bpt):
    return loss_nats * LOG2_E / bpt

@torch.inference_mode()
def evaluate_test(iface, data, batch_size=8):
    # Defensive: eval mode disables dropout/etc even if a stray .train()
    # was called by an earlier notebook cell.
    iface.model.eval()
    n = data.shape[0]
    # Accumulate the three metrics on-device — one .cpu() at the very
    # end instead of three .item() syncs per batch (drops the launch
    # queue ~750 times per eval otherwise).
    sum_loss = torch.zeros(1, device=device, dtype=torch.float64)
    sum_top1 = torch.zeros(1, device=device, dtype=torch.float64)
    sum_top5 = torch.zeros(1, device=device, dtype=torch.float64)
    n_batches = 0
    for i in tqdm(range(0, n, batch_size), desc=f'eval {iface.name}'):
        chunk = data[i:i+batch_size].to(device)
        if chunk.size(0) == 0:
            break
        x, y = chunk[:, :-1], chunk[:, 1:]
        logits = iface.forward(x)
        flat_l = logits.view(-1, logits.size(-1))
        flat_t = y.contiguous().view(-1)
        sum_loss += F.cross_entropy(flat_l, flat_t, reduction='mean').double()
        sum_top1 += (flat_l.argmax(-1) == flat_t).double().mean()
        topk = flat_l.topk(5, dim=-1).indices
        sum_top5 += (topk == flat_t.unsqueeze(-1)).any(-1).double().mean()
        n_batches += 1
    # One synchronisation point.
    avg = (torch.stack([sum_loss, sum_top1, sum_top5]) / max(n_batches, 1)).cpu().tolist()
    avg_loss, avg_top1, avg_top5 = avg[0][0], avg[1][0], avg[2][0]
    return {
        'loss':       avg_loss,
        'perplexity': math.exp(min(avg_loss, 20)),
        'bpb':        bits_per_byte(avg_loss, bytes_per_token),
        'top1':       avg_top1,
        'top5':       avg_top5,
    }

results_test = {}
for iface in (mamba_iface, xfmr_iface):
    print(f"\n══ {iface.name} ══")
    r = evaluate_test(iface, test_data, batch_size=EVAL_BATCH_SIZE)
    results_test[iface.name] = r
    for k, v in r.items():
        print(f"  {k:12s} = {v:.4f}")


In [ ]:
# Side-by-side table
df_test = pd.DataFrame(results_test).T
df_test.index.name = 'model'
df_test


## Experiment 2 — Inference benchmark

Single-stream (batch=1) measurements — what one serving user sees. Mamba's claims:
* O(1) per-token decode (no growing KV cache)
* Inference state size **constant in sequence length**

We report decode tok/s, prefill tok/s at multiple lengths, and analytical state size up to the longest sequence we'd realistically serve.


In [ ]:
def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Try to opt into torch.compile for the decode step. step() is a
# fixed-shape recurrent kernel called 100+ times — ideal for Inductor's
# kernel fusion. We DO NOT compile forward() (the prefill path) because
# it goes through `_XlaFusedSSM.apply` (a custom autograd.Function) which
# torch.compile may graph-break on — measure separately if you want it.
#
# `mode="default"` is the safe choice: kernel fusion + autotuning, but
# NO CUDA Graphs. The more aggressive `"reduce-overhead"` mode wraps
# the compiled function in CUDA Graphs which requires fully-static
# input/output identity — incompatible with our `(conv_state, ssm_state)`
# tuple cache (the tuple is rebuilt per call) and with the lazily-
# populated `_A_neg_exp_cache` Python attribute. If you want to try
# CUDA Graphs explicitly, use `USE_CUDA_GRAPHS = True` further below.
USE_TORCH_COMPILE = True            # set False to skip compile entirely

# Suppress Dynamo errors so any per-step recompile that hits a graph
# break degrades gracefully to eager instead of crashing the benchmark.
import torch._dynamo
torch._dynamo.config.suppress_errors = True

def maybe_compile_step(iface, mode="default"):
    if not (USE_TORCH_COMPILE and torch.cuda.is_available()):
        return iface           # CPU is too slow to amortize compile overhead
    target = getattr(iface, "model", None)
    if target is None or not hasattr(target, "step"):
        return iface
    original_step = target.step
    try:
        cache = target.allocate_inference_cache(
            batch_size=1, dtype=torch.float32, device=device,
        )
        dummy_in = torch.zeros(1, dtype=torch.long, device=device)
        # Phase 1: EAGER warmup. Populates lazy per-block state (the
        # `_A_neg_exp_cache` attribute on MambaBlock). Without this, the
        # first compiled call would see the attribute mutate from None
        # to a tensor — Dynamo tracks that as a guarded side-effect and
        # bails with the cryptic `weakref to UntypedStorage` error from
        # this user's traceback.
        with torch.inference_mode():
            _, cache = original_step(dummy_in, cache)
        # Phase 2: compile and warm the compiled graph (triggers Inductor
        # at predictable code instead of mid-benchmark).
        compiled = torch.compile(original_step, mode=mode, dynamic=False)
        with torch.inference_mode():
            _, cache = compiled(dummy_in, cache)   # graph capture
            _, cache = compiled(dummy_in, cache)   # cached replay
        target.step = compiled
        print(f"  compiled {iface.name}.step  (mode={mode})")
    except Exception as e:
        # Restore eager step. The benchmark still runs, just without the
        # compile speedup. This catches all the known fragility around
        # torch.compile + inference_mode + stateful caches.
        target.step = original_step
        msg = type(e).__name__ + ': ' + str(e)
        print(f"  torch.compile skipped for {iface.name}: {msg[:200]}")
    return iface


@torch.inference_mode()
def benchmark_inference(iface, prefill_lens, decode_tokens,
                        n_warmup=3, n_iter=20):
    """Single-stream inference benchmark.

    n_warmup=3 / n_iter=20 (vs the original 1/5) because torch_xla — and
    sometimes CUDA cuDNN benchmark mode — can re-trace on the second
    call as the runtime decides on a faster path; capturing even one
    cold-compile in a 5-iter window inflates measured latency by ~20%
    and produced the non-monotonic prefill-TPS curves we saw in the
    pre-fix runs.
    """
    iface.model.eval()
    out = {}
    # ── Prefill ──────────────────────────────────────────────────────────
    for L in prefill_lens:
        x = torch.zeros(1, L, dtype=torch.long, device=device)
        for _ in range(n_warmup):
            _ = iface.forward(x)
        synchronize()
        t0 = time.time()
        for _ in range(n_iter):
            _ = iface.forward(x)
        synchronize()
        elapsed = time.time() - t0
        out[f'prefill_L{L}_tps']        = (n_iter * L) / max(elapsed, 1e-9)
        out[f'prefill_L{L}_latency_ms'] = (elapsed / n_iter) * 1000

    # ── Decode (autoregressive) ─────────────────────────────────────────
    cache = iface.allocate_cache(batch_size=1)
    next_tok = torch.zeros(1, dtype=torch.long, device=device)
    for _ in range(n_warmup):
        _, cache = iface.step(next_tok, cache)
    synchronize()
    t0 = time.time()
    for _ in range(decode_tokens):
        _, cache = iface.step(next_tok, cache)
    synchronize()
    elapsed = time.time() - t0
    out['decode_tps']        = decode_tokens / max(elapsed, 1e-9)
    out['decode_latency_ms'] = (elapsed / decode_tokens) * 1000
    return out


def run_inference_benchmark_at_dtype(target_dtype, label):
    """Cast both models to `target_dtype` and run the benchmark.

    SSM state stays fp32 regardless (numerical stability). Linear weights
    follow `target_dtype`. Returns the same dict shape as the original
    benchmark, plus a 'dtype' key.
    """
    print(f"\n┌── inference benchmark @ dtype={target_dtype} ({label}) ──")
    # Cast models to the target dtype (cheap; weights only).
    mamba_iface.model.to(target_dtype)
    xfmr_iface.model.to(target_dtype)
    # SSM state cache must remain fp32; allocate_cache handles that
    # explicitly via dtype=torch.float32 in MambaInterface.allocate_cache.

    out = {}
    for iface in (mamba_iface, xfmr_iface):
        print(f"│  {iface.name} ...")
        r = benchmark_inference(iface, INFERENCE_PREFILL_LENS, INFERENCE_DECODE_TOKENS)
        r['dtype'] = str(target_dtype)
        out[iface.name] = r
        for k, v in r.items():
            print(f"│    {k:32s} = {v if isinstance(v, str) else f'{v:,.4g}'}")
    print(f"└── done @ {target_dtype}")
    return out


# Opt into torch.compile for the per-token recurrent step BEFORE the
# benchmark runs (the warmup pays the ~10-30s compile cost the first
# time it fires). Skip on CPU — compile overhead doesn't amortize.
mamba_iface = maybe_compile_step(mamba_iface)
xfmr_iface  = maybe_compile_step(xfmr_iface)

# CUDA Graphs (opt-in). Captures the entire per-token decode into a
# single CUDA graph and replays it. Stacks with torch.compile for
# another ~1.5-3x decode speedup at small batch on T4 (where kernel
# launch overhead dominates), but is fragile: requires fixed shapes,
# may not capture cleanly through HuggingFace's transformer past_kv
# bookkeeping, and needs RNG plumbing for sampling. Sampling stays
# outside the graph here. Flip this to True to try.
USE_CUDA_GRAPHS = False
if USE_CUDA_GRAPHS and torch.cuda.is_available():
    print("CUDA Graphs experimental path is enabled — see the wrapper for caveats.")
    # Minimal capture wrapper: pre-allocate one (B=1,) input id buffer,
    # warmup three steps, then capture step() into a graph. Replay copies
    # the real input id in, replays, reads the logits out. The cache
    # tensors are themselves pre-allocated by allocate_cache and are
    # mutated in-place by the captured kernels — only safe if step()
    # actually writes in-place to the cache, which is the case for
    # mamba.mamba_block.MambaBlock.step but NOT for HuggingFace
    # transformers (which return a new past_key_values tuple).
    def _try_capture_graph(iface):
        if iface.name != "Mamba":     # transformer past_kv tuple isn't graph-capturable as-is
            return iface
        cache = iface.allocate_cache(batch_size=1)
        static_in = torch.zeros(1, dtype=torch.long, device=device)
        # Warmup
        for _ in range(3):
            _, cache = iface.step(static_in, cache)
        torch.cuda.synchronize()
        try:
            g = torch.cuda.CUDAGraph()
            with torch.cuda.graph(g):
                static_logits, _ = iface.step(static_in, cache)
            class _Captured:
                name = iface.name
                model = iface.model
                cfg = iface.cfg
                allocate_cache = iface.allocate_cache
                state_size_bytes = iface.state_size_bytes
                n_params = iface.n_params
                forward = iface.forward
                def step(self, input_ids, _cache):
                    static_in.copy_(input_ids)
                    g.replay()
                    return static_logits, cache
            print(f"  captured CUDA graph for {iface.name}.step")
            return _Captured()
        except Exception as e:
            print(f"  CUDA graph capture failed for {iface.name}: {e!r}")
            return iface
    mamba_iface = _try_capture_graph(mamba_iface)
    xfmr_iface  = _try_capture_graph(xfmr_iface)

# Run TWICE: once at fp32 (matches the quality-metric configuration),
# once at the realistic deployment dtype. Both numbers go into the
# results JSON.
results_inf_fp32   = run_inference_benchmark_at_dtype(DTYPE,                    "fair-comparison")
results_inf_deploy = run_inference_benchmark_at_dtype(INFERENCE_DTYPE_DEPLOY,   "deployment")

# Restore models to the quality-metric dtype for downstream cells.
mamba_iface.model.to(DTYPE)
xfmr_iface.model.to(DTYPE)

# Use the fp32 measurements for the headline metrics (back-compat with
# downstream cells that index into `results_inf`); both dtypes are saved
# in the results JSON below.
results_inf = results_inf_fp32

# Free the temporary buffers / re-tracing artifacts before the next
# experiment.
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Inference state size at increasing context lengths.
state_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
state_data = []
for L in state_lens:
    state_data.append({
        'seq_len':  L,
        'Mamba (KB)':       mamba_iface.state_size_bytes(1, L) / 1024,
        'Transformer (KB)': xfmr_iface.state_size_bytes(1, L) / 1024,
    })
df_state = pd.DataFrame(state_data).set_index('seq_len')
df_state['Transformer / Mamba'] = df_state['Transformer (KB)'] / df_state['Mamba (KB)']
df_state


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4.2))
for col in ['Mamba (KB)', 'Transformer (KB)']:
    ax.plot(df_state.index, df_state[col], marker='o', label=col)
ax.set_xscale('log', base=2); ax.set_yscale('log', base=2)
ax.set_xlabel('Sequence length'); ax.set_ylabel('Inference state per stream (KB, fp32)')
ax.set_title('Inference state vs sequence length')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()


## Experiment 3 — Loss vs sequence length

Train both models on `seq_len=512` then evaluate at `L ∈ {128, 256, 512, 1024, 2048}`. Mamba should hold up at `L > 512` (its recurrence is content-agnostic). Transformer with positional encoding may degrade out-of-distribution.

We re-pack the test stream at each target `L` so each evaluation is on full-length contexts.


In [ ]:
def repack_at_length(stream, L, max_chunks=512):
    chunk_size = L + 1
    n = len(stream) // chunk_size
    n = min(n, max_chunks)
    if n < 1:
        return None
    return torch.tensor(stream[:n * chunk_size], dtype=torch.long).view(n, chunk_size)

len_results = {iface.name: {} for iface in (mamba_iface, xfmr_iface)}
for L in LONG_CONTEXT_LENS:
    data_L = repack_at_length(stream, L, max_chunks=256)
    if data_L is None:
        print(f"L={L}: insufficient data")
        continue
    print(f"\nL={L}  ({data_L.shape[0]} chunks)")
    for iface in (mamba_iface, xfmr_iface):
        bs = max(1, EVAL_BATCH_SIZE // (L // 256 + 1))
        try:
            r = evaluate_test(iface, data_L, batch_size=bs)
            len_results[iface.name][L] = r
            print(f"  {iface.name:12s}  loss={r['loss']:.4f}  ppl={r['perplexity']:.2f}  bpb={r['bpb']:.4f}")
        except Exception as e:
            print(f"  {iface.name:12s}  FAILED: {e!r}")
            len_results[iface.name][L] = None

df_len = pd.DataFrame({
    name: {L: r['bpb'] if r else None for L, r in d.items()}
    for name, d in len_results.items()
})
df_len.index.name = 'seq_len'
df_len


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4.2))
for col in df_len.columns:
    ax.plot(df_len.index, df_len[col], marker='o', label=col)
ax.axvline(SEQ_LEN, color='gray', linestyle='--', alpha=0.6, label=f'trained L={SEQ_LEN}')
ax.set_xscale('log', base=2)
ax.set_xlabel('Evaluation sequence length'); ax.set_ylabel('Bits per byte (lower is better)')
ax.set_title('Loss vs sequence length — extrapolation test')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Free the long-context (L=2048) buffers before the next experiment.
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Experiment 4 — Per-position loss

For position `t ∈ [0, seq_len)`, average the loss across all test sequences. A model that uses its context produces a monotonically decreasing curve. A flat curve = the model isn't accumulating context. A U-shape = something pathological.


In [ ]:
@torch.inference_mode()
def per_position_loss(iface, data, batch_size=8, max_chunks=200):
    iface.model.eval()
    n = min(data.shape[0], max_chunks)
    L_minus_1 = data.shape[1] - 1
    accum = torch.zeros(L_minus_1, device=device)
    count = 0
    for i in tqdm(range(0, n, batch_size), desc=f'per-pos {iface.name}'):
        chunk = data[i:i+batch_size].to(device)
        if chunk.size(0) == 0:
            break
        x, y = chunk[:, :-1], chunk[:, 1:]
        logits = iface.forward(x)                            # (B, L, V)
        # Per-position cross-entropy
        log_p = F.log_softmax(logits.float(), dim=-1)
        gather = log_p.gather(-1, y.unsqueeze(-1)).squeeze(-1)   # (B, L)
        nll = -gather                                            # (B, L)
        accum += nll.sum(0)
        count += chunk.size(0)
    return (accum / max(count, 1)).cpu().numpy()

per_pos = {}
for iface in (mamba_iface, xfmr_iface):
    per_pos[iface.name] = per_position_loss(iface, test_data, batch_size=EVAL_BATCH_SIZE)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4.2))
positions = np.arange(1, SEQ_LEN + 1)  # 1-indexed for display
for name, curve in per_pos.items():
    # Smooth slightly with a rolling window
    w = max(1, len(curve) // 50)
    smooth = np.convolve(curve, np.ones(w)/w, mode='same')
    ax.plot(positions, smooth, label=name, alpha=0.85)
ax.set_xlabel('Token position'); ax.set_ylabel('Mean cross-entropy (nats)')
ax.set_title('Per-position loss curve')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Slope of decrease (early vs late)
for name, curve in per_pos.items():
    early = float(curve[:50].mean())
    late  = float(curve[-50:].mean())
    print(f"{name:12s}  pos 0-50 mean = {early:.4f}   pos last-50 mean = {late:.4f}   "
          f"Δ = {late - early:+.4f}")


## Experiment 5 — Selective copying / induction heads

The classic test of in-context retrieval. We construct sequences of the form:

    [A] [B] [filler × n] [A] → score the model's P([B] | context)

where `[A]` is a marker token that appears twice and `[B]` is the target the model should "copy" from earlier in the context.

**Why this is hard for our 5M models.** Both Mamba and the transformer were trained on prose (SimpleStories), not on synthetic copy tasks. Pure induction-head emergence on random token IDs typically requires (a) explicit copy-task training or (b) much larger scale. So the absolute numbers will be small — but the **headline question is RELATIVE**: does Mamba's recurrence carry the marker→target association across longer gaps than the transformer's attention?

**Scoring methodology.** Top-1 over a 4 096-token vocabulary with n=50 trials cannot detect anything below ~2% absolute — a partially-working induction head reads as "0/50" indistinguishably from random. Headline metric is therefore **2AFC top-1**: given the prompt above, is the model's logit for the true target `[B]` higher than its logit for a random *distractor* (different target sampled identically)? Random baseline = **50%**, n=500 trials gives 95% CI half-width ≈ 4.4pp, so we can detect whether either model is meaningfully above chance. Full-vocab top-1 and log-prob-of-target are reported alongside as secondary signals.

**Token selection.** Markers and targets are both drawn from a *mid-frequency band* of the vocabulary (BPE-rank 200-1200 — neither rare-tail tokens nor the most-common stop words), so the model's prior over them is roughly uniform. Fillers come from the same band so the model can't trivially distinguish target from filler by frequency alone.


In [ ]:
import random

V = tokenizer.vocab_size
# All three pools come from the SAME mid-frequency band so the model
# can't discriminate by frequency rather than by induction. The previous
# split (markers from rare-tail, fillers from common-head, targets from
# mid) made the test answerable by "predict whatever's most common"
# regardless of context. BPE puts most-common tokens first, rare-tail
# last; the [200, 1200) band is the mid-frequency working vocabulary
# for SimpleStories-class tokenizers.
mid_band = list(range(min(200, V // 4), min(1200, 3 * V // 4)))
filler_pool = mid_band
marker_pool = mid_band
target_pool = mid_band

@torch.inference_mode()
def induction_score(iface, n_filler, n_trials=500):
    """Score in-context retrieval with three complementary metrics.

    Returns:
        top1_2afc:    fraction of trials where logit[target] > logit[distractor]
                      (50% baseline; statistically meaningful at n_trials=500)
        top1_full:    full-vocab argmax accuracy (1/V baseline; reported for
                      comparison with the original methodology — most likely
                      ~0% at this scale)
        log_p_target: mean log-probability assigned to the target token
        mean_rank:    mean rank of target across the full vocab (lower=better)
    """
    iface.model.eval()
    correct_2afc = 0
    correct_full = 0
    log_ps, ranks = [], []
    rng = random.Random(42 + n_filler)
    for _ in range(n_trials):
        marker, target, distractor = rng.sample(marker_pool, 3)
        # Distractor should be different from target AND a plausible
        # alternative — same pool, sampled at the same time.
        if distractor == target:                     # vanishingly rare; resample
            distractor = rng.choice([t for t in target_pool if t != target])
        fillers = [rng.choice(filler_pool) for _ in range(n_filler)]
        # Make sure neither target nor distractor accidentally appear in
        # the fillers — otherwise the "context" gives the model the
        # answer trivially or biases it toward distractor.
        fillers = [f for f in fillers if f != target and f != distractor]
        if len(fillers) < n_filler:
            extra_pool = [t for t in filler_pool if t != target and t != distractor]
            fillers += rng.sample(extra_pool, n_filler - len(fillers))

        seq = [marker, target] + fillers + [marker]
        ids = torch.tensor([seq], dtype=torch.long, device=device)
        logits = iface.forward(ids)[0, -1].float()    # logits AFTER last marker

        if logits[target] > logits[distractor]:
            correct_2afc += 1

        log_p = F.log_softmax(logits, dim=-1)
        log_ps.append(float(log_p[target]))
        rank = int((logits > logits[target]).sum())
        ranks.append(rank)
        if int(logits.argmax()) == target:
            correct_full += 1

    return {
        'top1_2afc':    correct_2afc / n_trials,    # PRIMARY: 50% = random
        'top1_full':    correct_full / n_trials,    # secondary: 1/V = random
        'log_p_target': float(np.mean(log_ps)),
        'mean_rank':    float(np.mean(ranks)),
    }

ind_results = {iface.name: {} for iface in (mamba_iface, xfmr_iface)}
for n in INDUCTION_DISTANCES:
    print(f"\nDistance n={n}:")
    for iface in (mamba_iface, xfmr_iface):
        r = induction_score(iface, n_filler=n, n_trials=INDUCTION_TRIALS)
        ind_results[iface.name][n] = r
        print(f"  {iface.name:12s}  2AFC={r['top1_2afc']*100:5.1f}%  "
              f"full-top1={r['top1_full']*100:5.2f}%  "
              f"log_p={r['log_p_target']:7.3f}  rank={r['mean_rank']:6.0f}")

# 2AFC headline table (50% baseline; reported in main results).
df_ind_2afc = pd.DataFrame({
    name: {n: r['top1_2afc'] for n, r in d.items()} for name, d in ind_results.items()
})
df_ind_2afc.index.name = 'distance'
print('\n2AFC accuracy (random baseline = 50%):')
df_ind_2afc


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# 2AFC accuracy vs distance (PRIMARY headline metric).
# 95% Wald CI half-width = 1.96 * sqrt(p*(1-p)/n_trials); plot it as a band.
ci_halfwidth = 1.96 * (0.5 * 0.5 / INDUCTION_TRIALS) ** 0.5
for col in df_ind_2afc.columns:
    p = df_ind_2afc[col].values
    axes[0].plot(df_ind_2afc.index, p * 100, marker='o', label=col)
    axes[0].fill_between(df_ind_2afc.index,
                         (p - ci_halfwidth) * 100,
                         (p + ci_halfwidth) * 100,
                         alpha=0.15)
axes[0].axhline(50.0, color='gray', linestyle='--', alpha=0.6, label='random (50%)')
axes[0].set_xscale('log', base=2)
axes[0].set_xlabel('Filler distance (n)')
axes[0].set_ylabel('2AFC accuracy (%)')
axes[0].set_title('Selective copying — 2AFC vs distance (random=50%)')
axes[0].set_ylim(40, 100)
axes[0].grid(True, which='both', alpha=0.3); axes[0].legend()

# Mean rank of target — secondary signal. Lower = the model assigned the
# target a higher logit relative to the rest of the vocab.
df_ind_rank = pd.DataFrame({
    name: {n: r['mean_rank'] for n, r in d.items()} for name, d in ind_results.items()
})
for col in df_ind_rank.columns:
    axes[1].plot(df_ind_rank.index, df_ind_rank[col] + 1, marker='o', label=col)
axes[1].axhline(tokenizer.vocab_size / 2, color='gray', linestyle='--', alpha=0.6,
                label=f'random (V/2 = {tokenizer.vocab_size//2})')
axes[1].set_xscale('log', base=2); axes[1].set_yscale('log', base=2)
axes[1].set_xlabel('Filler distance (n)')
axes[1].set_ylabel('Mean rank of target token (lower better)')
axes[1].set_title('Selective copying — rank of target')
axes[1].grid(True, which='both', alpha=0.3); axes[1].legend()

plt.tight_layout(); plt.show()


## Experiment 6 — Diversity analysis

Generate `N` completions from each prompt and compute:
* **Self-BLEU** — how similar are the N generations to each other (lower = more diverse)
* **Distinct-1, distinct-2** — fraction of unique unigrams / bigrams (higher = more diverse)
* **Repetition rate** — fraction of bigrams that appear more than once (lower = less degenerate)

A model that always produces the same completion has self-BLEU = 1.0 and very low distinct-n. Detects mode collapse the loss number can't see.


In [ ]:
@torch.inference_mode()
def generate(iface, prompt_ids, max_new_tokens,
             temperature=0.8, top_k=40, top_p=None):
    """Gumbel-max sampling — works for any unified interface.

    Supports top-k AND nucleus (top-p) filtering. Token IDs stay on-device
    throughout the loop and transfer to host in a single `.cpu().tolist()`
    at the end (vs the original per-step `int(next_tok)` host syncs).

    Args:
        temperature: > 0; effectively greedy when `top_k=1`.
        top_k:       keep top-k logits (None or 0 = disable).
        top_p:       nucleus threshold in (0, 1] (None or 1 = disable).
                     Applied AFTER top-k if both are set.
    """
    iface.model.eval()
    cache = iface.allocate_cache(batch_size=1)
    # Prefill: feed prompt tokens one at a time through step()
    for tok in prompt_ids:
        _, cache = iface.step(tok.unsqueeze(0), cache)
    out_buf = []
    next_tok = prompt_ids[-1].unsqueeze(0)
    for _ in range(max_new_tokens):
        logits, cache = iface.step(next_tok, cache)
        logits = logits / max(temperature, 1e-6)
        if top_k:
            v, _ = torch.topk(logits, k=min(top_k, logits.shape[-1]), dim=-1)
            logits = torch.where(logits < v[:, [-1]],
                                 torch.full_like(logits, float('-inf')), logits)
        if top_p is not None and top_p < 1.0:
            # Nucleus sampling (Holtzman et al., ICLR 2020): keep the
            # smallest set whose cumulative probability exceeds top_p,
            # always retaining at least the top-1 token.
            sorted_logits, sorted_idx = torch.sort(logits, dim=-1, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            mask = cum_probs > top_p
            mask[:, 0] = False                   # always keep top-1
            sorted_logits = sorted_logits.masked_fill(mask, float('-inf'))
            logits = torch.empty_like(logits).scatter_(-1, sorted_idx, sorted_logits)
        u = torch.rand_like(logits).clamp(min=1e-9)
        next_tok = torch.argmax(logits + (-torch.log(-torch.log(u))), dim=-1)
        out_buf.append(next_tok)
    out_ids = torch.cat(out_buf, dim=0).cpu().tolist()
    if eos_id in out_ids:
        out_ids = out_ids[:out_ids.index(eos_id)]
    return prompt_ids.tolist() + out_ids

print("Generating samples — this can take a few minutes per model...")
generations = {iface.name: {p: [] for p in DIVERSITY_PROMPTS} for iface in (mamba_iface, xfmr_iface)}

for iface in (mamba_iface, xfmr_iface):
    for prompt in DIVERSITY_PROMPTS:
        pids = torch.tensor(tokenizer.encode(prompt, add_special_tokens=False),
                            dtype=torch.long, device=device)
        if pids.numel() == 0:
            pids = torch.tensor([eos_id], dtype=torch.long, device=device)
        for trial in tqdm(range(DIVERSITY_GENS_PER_PROMPT),
                          desc=f'{iface.name}: {prompt!r}'):
            gen_ids = generate(iface, pids, DIVERSITY_MAX_NEW_TOKENS,
                               temperature=DIVERSITY_TEMPERATURE,
                               top_k=DIVERSITY_TOP_K)
            text = tokenizer.decode(gen_ids, skip_special_tokens=True)
            generations[iface.name][prompt].append(text)

# Show one example per prompt per model
for name, by_prompt in generations.items():
    print(f"\n══ {name} sample ══")
    for prompt, texts in list(by_prompt.items())[:2]:
        print(f"  [{prompt!r}] → {texts[0]!r}")


In [ ]:
def compute_diversity(texts):
    """Returns dict with self-BLEU, distinct-1, distinct-2, repetition_rate."""
    tokenised = [t.split() for t in texts]

    # Self-BLEU: each text scored against the others as references
    smooth = SmoothingFunction().method1
    bleus = []
    for i, hyp in enumerate(tokenised):
        refs = [t for j, t in enumerate(tokenised) if j != i]
        if not refs or not hyp:
            continue
        bleus.append(sentence_bleu(refs, hyp, smoothing_function=smooth))
    self_bleu = float(np.mean(bleus)) if bleus else 0.0

    # Distinct-n
    all_uni = [tok for txt in tokenised for tok in txt]
    all_bi  = [bg for txt in tokenised for bg in zip(txt[:-1], txt[1:])]
    distinct_1 = len(set(all_uni)) / max(len(all_uni), 1)
    distinct_2 = len(set(all_bi))  / max(len(all_bi), 1)

    # Repetition rate: per generation, fraction of bigrams that repeat
    rep = []
    for txt in tokenised:
        if len(txt) < 2: continue
        bgs = list(zip(txt[:-1], txt[1:]))
        c = Counter(bgs)
        rep.append(sum(v for v in c.values() if v > 1) / len(bgs))
    repetition = float(np.mean(rep)) if rep else 0.0

    return {
        'self_bleu':      self_bleu,
        'distinct_1':     distinct_1,
        'distinct_2':     distinct_2,
        'repetition_rate': repetition,
    }

div_results = {}
for name, by_prompt in generations.items():
    per_prompt_metrics = {p: compute_diversity(texts) for p, texts in by_prompt.items()}
    # Average across prompts
    keys = ['self_bleu', 'distinct_1', 'distinct_2', 'repetition_rate']
    div_results[name] = {k: float(np.mean([m[k] for m in per_prompt_metrics.values()]))
                         for k in keys}

df_div = pd.DataFrame(div_results).T
df_div.index.name = 'model'
df_div


## Experiment 7 — Long-form generation quality

The diversity benchmark in Experiment 6 caps generation at 80 tokens. This experiment pushes both models to **1024 tokens** — *2× the trained context length* — and measures four complementary aspects of the long-form output:

1. **MAUVE** (Pillutla et al., NeurIPS 2021 outstanding paper, [arxiv:2102.01454](https://arxiv.org/abs/2102.01454)) — distributional distance between model generations and held-out human stories, measured in a GPT-2 embedding space. *Higher is better; 1.0 = indistinguishable from human.* The headline cross-architecture signal.
2. **rep-4** (Welleck et al., ICLR 2020 [arxiv:1908.04319](https://arxiv.org/abs/1908.04319)) — fraction of 4-grams that repeat in the generation. *Lower is better; humans ≈ 0.005, mode-collapsed greedy LMs > 0.10.* Catches the "and then and then and then" failure mode that distinct-n misses.
3. **Sentence-embedding drift slope** — the slope of `cos(sentence_t, sentence_0)` across the generation. *Slope ≈ 0 = on-topic; strongly negative = drifts away.* This is where the Mamba length-extrapolation claim becomes visible: Mamba's recurrent state should hold the prompt's topic better at long generations than the transformer's RoPE-extrapolated attention.
4. **Pairwise judge** (Qwen2.5-3B-Instruct, double-ordered to mitigate position bias per [arxiv:2506.22316](https://arxiv.org/html/2506.22316v1)) — open-source LM-as-judge graded on the Eldan & Li 2023 TinyStories rubric (grammar / coherence / creativity). *Reports double-confirmed win-rate.* Optional — gated on `RUN_JUDGE` because the 3B judge takes ~10 minutes for the full pairing.

**Generation regime:** two sampling modes per model — *nucleus* (`temperature=0.8, top_p=0.95`, the deployment-realistic setting MAUVE was designed for) and *greedy* (the worst-case for degeneration; rep-4 is most informative here). Slice each generation at positions {256, 512, 1024} so we can plot quality-vs-length curves and ask the central question: *do the metrics get worse past the trained context (L=512), and if so, which model degrades faster?*

**Time budget on Colab T4** ≈ 30–40 min total (generations dominate), or ~10 min if `RUN_JUDGE = False`.


In [ ]:
# ─── Long-form generation config ───────────────────────────────────────
LF_N_PROMPTS    = 20         # 4 hand-picked + 16 from held-out test stories
LF_GENS_PER_P   = 2          # 2 samples per prompt per mode
LF_MAX_NEW      = 1024       # 2× trained context — the extrapolation test
LF_LEN_SLICES   = [256, 512, 1024]
LF_MODES        = {
    "nucleus": dict(temperature=0.8, top_p=0.95, top_k=0),
    "greedy":  dict(temperature=1e-6, top_p=None, top_k=1),
}
RUN_JUDGE       = USE_GPU    # judge is 3B params — needs CUDA to be tractable
JUDGE_MODEL     = "Qwen/Qwen2.5-3B-Instruct"

# Build prompt set: hand-picked seeds + held-out story openers. The
# `ds_tok` test stream already burned the first MAX_TEST_STORIES; we
# pick the LAST stories from `ds` (still in test split) for prompts and
# MAUVE references so there's no overlap between training the metrics
# and computing them.
n_extra = LF_N_PROMPTS - len(DIVERSITY_PROMPTS)
holdout_stories = ds[TEXT_COLUMN][-(n_extra + LF_N_PROMPTS * LF_GENS_PER_P):]
extra_prompts = [s.split('.')[0].strip().lower() + '.'
                 for s in holdout_stories[:n_extra]
                 if s.split('.')[0].strip()]
LF_PROMPTS = (DIVERSITY_PROMPTS + extra_prompts)[:LF_N_PROMPTS]
human_refs = [s.lower() for s in holdout_stories[n_extra:]]
print(f"Long-form prompts: {len(LF_PROMPTS)}")
print(f"MAUVE human references: {len(human_refs)}")


In [ ]:
# ─── Generate long-form samples (the slow step) ────────────────────────
@torch.inference_mode()
def generate_long(iface, prompt, mode):
    iface.model.eval()
    pids = torch.tensor(tokenizer.encode(prompt, add_special_tokens=False),
                        dtype=torch.long, device=device)
    if pids.numel() == 0:
        pids = torch.tensor([eos_id], dtype=torch.long, device=device)
    full_ids = generate(iface, pids, LF_MAX_NEW, **LF_MODES[mode])
    text = tokenizer.decode(full_ids, skip_special_tokens=True)
    return {"prompt": prompt, "ids": full_ids, "text": text,
            "n_new": len(full_ids) - len(pids.tolist())}

lf_gens = {iface.name: {m: [] for m in LF_MODES}
           for iface in (mamba_iface, xfmr_iface)}
for iface in (mamba_iface, xfmr_iface):
    for mode in LF_MODES:
        desc = f"{iface.name}/{mode}"
        for prompt in tqdm(LF_PROMPTS, desc=desc):
            for _ in range(LF_GENS_PER_P):
                lf_gens[iface.name][mode].append(generate_long(iface, prompt, mode))

# Show one sample per (model, mode) for a sanity check.
for name, by_mode in lf_gens.items():
    for mode, recs in by_mode.items():
        snippet = recs[0]['text'][:240].replace('\n', ' ')
        print(f"\n══ {name} / {mode} ({recs[0]['n_new']} new tokens) ══")
        print(f"  {snippet}{'…' if len(recs[0]['text']) > 240 else ''}")


In [ ]:
# ─── Metric 1+2: rep-4 + sentence drift, sliced by length ──────────────
from sentence_transformers import SentenceTransformer

st_enc = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))
print(f"Loaded SentenceTransformer (all-MiniLM-L6-v2) on {device}")

def rep_n(token_ids, n=4):
    """Welleck et al. ICLR 2020 rep-n on token IDs (not whitespace)."""
    if len(token_ids) < n:
        return 0.0
    grams = list(zip(*[token_ids[i:] for i in range(n)]))
    counts = Counter(grams)
    return sum(c for c in counts.values() if c > 1) / len(grams)

def drift_slope(text):
    """Slope of cos(sentence_t, sentence_0) across the generation.

    Negative slope ⇒ model is drifting away from the prompt's topic.
    Returns None if the generation is too short to fit a slope.
    """
    sents = nltk.sent_tokenize(text)
    if len(sents) < 4:
        return None
    embs = st_enc.encode(sents, normalize_embeddings=True,
                         show_progress_bar=False)
    sims_to_start = [float(np.dot(embs[i], embs[0])) for i in range(1, len(embs))]
    return float(np.polyfit(np.arange(len(sims_to_start)), sims_to_start, 1)[0])

def slice_metrics(recs, slice_tok):
    rep4_vals, drift_vals = [], []
    for r in recs:
        sliced_ids = r['ids'][:slice_tok]
        sliced_text = tokenizer.decode(sliced_ids, skip_special_tokens=True)
        rep4_vals.append(rep_n(sliced_ids, n=4))
        d = drift_slope(sliced_text)
        if d is not None:
            drift_vals.append(d)
    return {
        'rep4':        float(np.mean(rep4_vals)),
        'rep4_std':    float(np.std(rep4_vals)),
        'drift_slope': float(np.mean(drift_vals)) if drift_vals else float('nan'),
        'drift_std':   float(np.std(drift_vals))  if drift_vals else float('nan'),
        'n':           len(rep4_vals),
    }

degen_results = {iface.name: {m: {} for m in LF_MODES}
                 for iface in (mamba_iface, xfmr_iface)}
for iface in (mamba_iface, xfmr_iface):
    for mode in LF_MODES:
        for L in LF_LEN_SLICES:
            degen_results[iface.name][mode][L] = slice_metrics(
                lf_gens[iface.name][mode], L,
            )

# Print a compact summary table.
rows = []
for name in degen_results:
    for mode in LF_MODES:
        for L in LF_LEN_SLICES:
            r = degen_results[name][mode][L]
            rows.append({
                'model': name, 'mode': mode, 'slice_tok': L,
                'rep4':        f"{r['rep4']:.3f}",
                'drift_slope': f"{r['drift_slope']:+.4f}",
            })
df_degen = pd.DataFrame(rows)
df_degen


In [ ]:
# Plot: rep-4 and drift slope vs generation length, per (model, mode).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
markers = {'Mamba': 'o', 'Transformer': 's'}
linestyles = {'nucleus': '-', 'greedy': '--'}

for name in degen_results:
    for mode in LF_MODES:
        rep4 = [degen_results[name][mode][L]['rep4'] for L in LF_LEN_SLICES]
        drift = [degen_results[name][mode][L]['drift_slope'] for L in LF_LEN_SLICES]
        axes[0].plot(LF_LEN_SLICES, rep4,
                     marker=markers[name], linestyle=linestyles[mode],
                     label=f'{name} / {mode}', alpha=0.85)
        axes[1].plot(LF_LEN_SLICES, drift,
                     marker=markers[name], linestyle=linestyles[mode],
                     label=f'{name} / {mode}', alpha=0.85)

axes[0].axhline(0.05, color='gray', linestyle=':', alpha=0.6,
                label='healthy threshold (rep4≈0.05)')
axes[0].axvline(SEQ_LEN, color='red', linestyle=':', alpha=0.4,
                label=f'trained L={SEQ_LEN}')
axes[0].set_xlabel('Generation length (tokens)')
axes[0].set_ylabel('rep-4 (lower is healthier)')
axes[0].set_title('Degeneration (rep-4) vs generation length')
axes[0].grid(True, alpha=0.3); axes[0].legend(fontsize=9)

axes[1].axhline(0.0, color='gray', linestyle=':', alpha=0.6,
                label='no drift')
axes[1].axvline(SEQ_LEN, color='red', linestyle=':', alpha=0.4)
axes[1].set_xlabel('Generation length (tokens)')
axes[1].set_ylabel('Sentence drift slope (more negative = drifting)')
axes[1].set_title('Topic drift vs generation length')
axes[1].grid(True, alpha=0.3); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ─── Metric 3: MAUVE (distributional quality vs human) ─────────────────
import mauve

def mauve_score(model_texts, human_texts, label):
    if len(model_texts) < 8 or len(human_texts) < 8:
        print(f"  MAUVE skipped for {label}: need >=8 samples per side")
        return float('nan')
    try:
        out = mauve.compute_mauve(
            p_text=human_texts, q_text=model_texts,
            featurize_model_name='gpt2',
            max_text_length=LF_MAX_NEW,
            device_id=0 if torch.cuda.is_available() else -1,
            verbose=False,
        )
        return float(out.mauve)
    except Exception as e:
        print(f"  MAUVE failed for {label}: {e!r}")
        return float('nan')

mauve_results = {}
for name, by_mode in lf_gens.items():
    mauve_results[name] = {}
    for mode, recs in by_mode.items():
        texts = [r['text'] for r in recs]
        score = mauve_score(texts, human_refs, f"{name}/{mode}")
        mauve_results[name][mode] = score
        print(f"  MAUVE  {name:12s} {mode:8s} = {score:.3f}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ─── Metric 4: Pairwise LLM-as-judge (open-source, double-ordered) ────
judge_results = {mode: None for mode in LF_MODES}

if RUN_JUDGE:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import json as _json

    print(f"Loading judge: {JUDGE_MODEL}")
    jtok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
    # Try 4-bit first (saves ~4GB on T4); fall back to bf16 if bnb is broken.
    try:
        from transformers import BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_4bit=True,
                                 bnb_4bit_compute_dtype=torch.bfloat16)
        jmodel = AutoModelForCausalLM.from_pretrained(
            JUDGE_MODEL, quantization_config=bnb, device_map=str(device),
        ).eval()
        print("  loaded judge in 4-bit")
    except Exception as e:
        print(f"  4-bit load failed ({e!r}); retrying bf16")
        jmodel = AutoModelForCausalLM.from_pretrained(
            JUDGE_MODEL, torch_dtype=torch.bfloat16,
        ).to(device).eval()

    RUBRIC = (
        "You are grading two short children's-story continuations of the same "
        "prompt. Pick the better one on three axes:\n"
        "  - grammar:   correctness of English\n"
        "  - coherence: stable characters, no contradictions, on-topic\n"
        "  - creativity: interesting / not generic\n"
        "Reply with ONLY one valid JSON object, nothing else, in this exact "
        "schema (use \"A\", \"B\", or \"tie\" for each axis):\n"
        '{"winner":"A|B|tie","grammar":"A|B|tie","coherence":"A|B|tie","creativity":"A|B|tie"}'
    )
    JUDGE_TRUNC = 1500   # chars; verbosity-bias mitigation

    @torch.inference_mode()
    def judge_pair(prompt, text_a, text_b):
        msgs = [{"role": "system", "content": RUBRIC},
                {"role": "user",
                 "content": (f"PROMPT:\n{prompt}\n\n"
                             f"A:\n{text_a[:JUDGE_TRUNC]}\n\n"
                             f"B:\n{text_b[:JUDGE_TRUNC]}")}]
        ids = jtok.apply_chat_template(msgs, return_tensors='pt',
                                       add_generation_prompt=True).to(device)
        out = jmodel.generate(ids, max_new_tokens=80, do_sample=False,
                              pad_token_id=jtok.eos_token_id)
        txt = jtok.decode(out[0, ids.shape[-1]:], skip_special_tokens=True)
        try:
            j_open  = txt.find('{')
            j_close = txt.rfind('}')
            return _json.loads(txt[j_open:j_close + 1])
        except Exception:
            return None

    def double_ordered_winrate(gens_a, gens_b, label):
        """Run each pair in both orderings; only count A as winning if it
        wins in BOTH (model-A in slot A) AND (model-A in slot B). This
        cancels the well-known position bias of small judges."""
        AXES = ('winner', 'grammar', 'coherence', 'creativity')
        tally = {ax: Counter() for ax in AXES}
        n_pairs = min(len(gens_a), len(gens_b))
        for i in tqdm(range(n_pairs), desc=f'judge {label}'):
            a, b = gens_a[i], gens_b[i]
            v1 = judge_pair(a['prompt'], a['text'], b['text'])
            v2 = judge_pair(a['prompt'], b['text'], a['text'])
            if v1 is None or v2 is None:
                for ax in AXES: tally[ax]['parse_fail'] += 1
                continue
            for ax in AXES:
                if v1.get(ax) == 'A' and v2.get(ax) == 'B':
                    tally[ax]['model_a_wins'] += 1
                elif v1.get(ax) == 'B' and v2.get(ax) == 'A':
                    tally[ax]['model_b_wins'] += 1
                else:
                    tally[ax]['tie_or_inconsistent'] += 1
        return {ax: dict(v) for ax, v in tally.items()}

    for mode in LF_MODES:
        m_recs = lf_gens['Mamba'][mode]
        x_recs = lf_gens['Transformer'][mode]
        judge_results[mode] = double_ordered_winrate(m_recs, x_recs, mode)
        print(f"\n══ judge / {mode} (Mamba=A, Transformer=B) ══")
        for ax, t in judge_results[mode].items():
            print(f"  {ax:11s} {t}")

    # Free the judge.
    del jmodel
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("RUN_JUDGE=False — skipping pairwise judge.")


In [ ]:
# Consolidated long-form results.
lf_summary = {}
for mode in LF_MODES:
    lf_summary[mode] = {
        'mauve':            {n: mauve_results[n][mode] for n in mauve_results},
        'rep4_at_1024':     {n: degen_results[n][mode][LF_LEN_SLICES[-1]]['rep4']
                             for n in degen_results},
        'drift_at_1024':    {n: degen_results[n][mode][LF_LEN_SLICES[-1]]['drift_slope']
                             for n in degen_results},
    }
    if judge_results.get(mode):
        lf_summary[mode]['judge'] = judge_results[mode]
df_lf = pd.DataFrame({mode: lf_summary[mode] for mode in LF_MODES}).T
df_lf


## Summary

Bring the headline numbers into one table. The interpretation:

* `bpb` and `top1/top5` — **lower bpb is better**, **higher top-k is better**. Quality.
* `decode_latency_ms` and `prefill_L*_tps` — **decode is Mamba's selling point**. Lower latency, higher tps = faster serving.
* `state_kb_at_L=512` — **lower is better at fixed L**, but the Mamba advantage compounds at long L (see Experiment 2 plot).
* `induction_2afc_at_n=64` and `=256` — **higher is better; 50% = random**. Tests architectural capability for in-context retrieval. Above 55% is meaningfully above chance at n=500 trials. The relative gap between Mamba and the Transformer at the longest distance is the key signal.
* `self_bleu` — **lower is more diverse**. `distinct_2` — **higher is more diverse**. `repetition_rate` — **lower is healthier**.
* `mauve_nucleus`, `rep4_at_1024`, `drift_at_1024` — long-form generation quality. **MAUVE higher better**, **rep-4 lower better**, **drift-slope nearer to 0 better**. The story for the writeup is the *trajectory* across slice lengths {256, 512, 1024} — does Mamba degrade past the trained context (L=512) more or less than the transformer?


In [ ]:
def safe_get(d, *keys, default=float('nan')):
    for k in keys:
        if d is None or k not in d:
            return default
        d = d[k]
    return d

summary = {}
for iface in (mamba_iface, xfmr_iface):
    name = iface.name
    summary[name] = {
        'n_params':                 iface.n_params,
        'test_loss':                results_test[name]['loss'],
        'test_perplexity':          results_test[name]['perplexity'],
        'test_bpb':                 results_test[name]['bpb'],
        'test_top1':                results_test[name]['top1'],
        'test_top5':                results_test[name]['top5'],
        'decode_tps':               results_inf[name]['decode_tps'],
        'decode_latency_ms':        results_inf[name]['decode_latency_ms'],
        f'prefill_L512_tps':        results_inf[name].get('prefill_L512_tps', float('nan')),
        f'state_kb_at_L=512':       iface.state_size_bytes(1, 512) / 1024,
        f'state_kb_at_L=2048':      iface.state_size_bytes(1, 2048) / 1024,
        f'induction_2afc_at_n=64':  safe_get(ind_results.get(name, {}).get(64), 'top1_2afc'),
        f'induction_2afc_at_n=256': safe_get(ind_results.get(name, {}).get(256), 'top1_2afc'),
        'diversity_self_bleu':      div_results[name]['self_bleu'],
        'diversity_distinct_2':     div_results[name]['distinct_2'],
        'diversity_repetition':     div_results[name]['repetition_rate'],
        # Long-form (Experiment 7) — nucleus mode at full 1024 tokens.
        'longform_mauve_nucleus':   safe_get(mauve_results, name, 'nucleus'),
        'longform_rep4_at_1024':    safe_get(degen_results, name, 'nucleus',
                                             LF_LEN_SLICES[-1], 'rep4'),
        'longform_drift_at_1024':   safe_get(degen_results, name, 'nucleus',
                                             LF_LEN_SLICES[-1], 'drift_slope'),
    }

df_summary = pd.DataFrame(summary).T
df_summary.index.name = 'model'
df_summary


In [ ]:
# Save the consolidated results to JSON for the writeup.
out = {
    'config': {
        'mamba_checkpoint':    MAMBA_CHECKPOINT_PATH,
        'transformer':         HF_TRANSFORMER_MODEL,
        'dataset':             DATASET_NAME,
        'split':               DATASET_SPLIT,
        'lowercase':           LOWERCASE,
        'seq_len':             SEQ_LEN,
        'n_test_stories':      len(ds),
        'bytes_per_token':     bytes_per_token,
    },
    'experiment_1_test':       results_test,
    'experiment_2_inference':  results_inf,             # fp32 (back-compat)
    'experiment_2_inference_fp32':   results_inf_fp32,
    'experiment_2_inference_deploy': results_inf_deploy,
    'experiment_2_state':      df_state.reset_index().to_dict(orient='records'),
    'experiment_3_loss_vs_len': len_results,
    'experiment_4_per_pos':    {k: v.tolist() for k, v in per_pos.items()},
    'experiment_5_induction':  ind_results,
    'experiment_6_diversity':  div_results,
    'experiment_7_longform':   {
        'config': {
            'n_prompts':     LF_N_PROMPTS,
            'gens_per_prompt': LF_GENS_PER_P,
            'max_new_tokens': LF_MAX_NEW,
            'modes':         {k: v for k, v in LF_MODES.items()},
            'judge_model':   JUDGE_MODEL if RUN_JUDGE else None,
        },
        'mauve':       mauve_results,
        'degeneration': degen_results,
        'judge':       judge_results,
        'summary':     lf_summary,
    },
    'summary':                 summary,
}
out_path = 'simplestories_eval_results.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"Wrote {out_path}")


---

**Done.** The results are in `simplestories_eval_results.json` plus the dataframes above.

For the writeup, the natural three-act structure is:
1. **Quality**: Experiments 1, 6, 7 — does Mamba match the transformer at fixed param budget on test BPB, generation diversity, and long-form metrics (MAUVE / rep-4 / drift / pairwise judge)?
2. **Efficiency**: Experiment 2 — does Mamba deliver the predicted serving advantages (decode tps + constant-size state)?
3. **Why**: Experiments 3, 4, 5, and the length-slice trajectory in 7 — long-context extrapolation, context use, in-context retrieval — explain the architectural differences. Experiment 7's slice-at-{256, 512, 1024} curves are the qualitative analog of Experiment 3's loss-vs-length: does the long-form quality degrade past the trained context, and if so, which model degrades faster?
